# 📊 00 · 统计推断与假设检验

> 秋招算法岗高频：**从样本打量总体**——点估计、区间估计、假设检验、功效、多重比较。

本节风格：公式 → numpy 手写 → `scipy.stats` 对照验证。理解每一行统计量背后的分布与自由度。

## 📋 本节要解决的问题

| # | 问题 | 工具 |
|---|---|---|
| 1 | 样本统计量能代表总体吗？ | CLT、抽样分布 |
| 2 | 「显著」是什么意思？ | 假设检验框架 |
| 3 | 用 z 还是 t？ | 方差已知与否、样本量 |
| 4 | 分类数据有关联吗？ | χ² 检验 |
| 5 | 多组均值有无差异？ | ANOVA F 检验 |
| 6 | 实验要做多大？ | 功效分析与样本量 |
| 7 | 检验做多了误报怎么办？ | Bonferroni / BH-FDR |

## 1️⃣ 统计推断基础

- **总体 vs 样本**：总体参数（μ, σ²）未知，我们用样本统计量（$\bar x$，$s^2$）估计；
- **点估计性质**：无偏（$E[\hat \theta]=\theta$）、一致（n→∞ 收敛）、有效（方差最小）；
- $s^2 = \dfrac{1}{n-1}\sum (x_i-\bar x)^2$ 的无偏性：分母 n-1 补偿「均值也由样本估计」的自由度损失；
- **标准误 SE**：统计量的标准差。$SE(\bar x)=\sigma/\sqrt n$，未知时用 $s/\sqrt n$。

> 面试点：为什么除 n-1？——样本方差中 $\bar x$ 由同一批数据算出，自由度剩 n-1。

## 2️⃣ 中心极限定理（CLT）

样本均值（n 独立同分布、方差有限）近似服从正态：

$$ \bar X \ \dot\sim\ \mathcal{N}\left(\mu,\ \frac{\sigma^2}{n}\right) $$

- 成立**不要求总体正态**，只要求独立同分布 + 方差有限；
- n 越大越准（经验法则 n ≥ 30，偏态时要更大）；
- 用途：构造置信区间、z 检验、AB 实验的均值指标。

下面用**指数分布**（严重偏态）验证：样本均值分布逼近正态。

In [1]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

N, n = 10000, 40
rng = np.random.RandomState(0)
means = rng.exponential(scale=2.0, size=(N, n)).mean(axis=1)   # 总体 μ=2, σ=2
t = np.linspace(means.min(), means.max(), 200)
plt.figure(figsize=(8, 3.6))
plt.hist(means, bins=60, density=True, alpha=0.6, label='样本均值分布')
plt.plot(t, stats.norm.pdf(t, 2.0, 2.0/np.sqrt(n)), 'r-', lw=2, label='理论 N(2, σ/√n)')
plt.xlabel('样本均值'); plt.ylabel('密度'); plt.legend()
plt.title('CLT：偏态指数总体的样本均值 → 正态')
plt.tight_layout(); plt.show()
print('样本均值 std = %.4f | 理论 σ/√n = %.4f' % (means.std(ddof=1), 2.0/np.sqrt(n)))

样本均值 std = 0.3151 | 理论 σ/√n = 0.3162


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2170134065.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3️⃣ 假设检验框架

- **H₀ 原假设**（无效应，如 μ=0） vs **H₁ 备择**（有效应）；
- **检验统计量**（z/t/F/χ²）在 H₀ 下的分布已知 → 算 **p 值** = 「H₀ 为真时看到当前或更极端结果」的概率；
- p < α（通常 0.05）→ **拒绝 H₀**，称显著；
- **两类错误**：第 I 类 α（误报，拒绝真 H₀）、第 II 类 β（漏报）；**功效** = 1-β = 正确拒绝假 H₀ 的概率；
- 决策非黑即白：p=0.049 与 0.051 的差别远小于它们与「效应量」的差别。

## 4️⃣ z 检验 vs t 检验

单样本均值检验：

$$ z = \frac{\bar x - \mu_0}{\sigma/\sqrt n}\ \ (\sigma\ \text{已知})\qquad t = \frac{\bar x - \mu_0}{s/\sqrt n}\ (\sigma\ \text{未知}) $$

- **z**：方差已知 → 标准正态；**t**：方差未知用样本 s → 学生 t 分布，**自由度 n-1**；
- n 大时 t → 正态，两者趋同（大样本 AB 实验常用 z/正态近似）；
- 双样本：**合并方差 t**（方差齐假定）vs **Welch t**（方差不齐，自由度不可整）；scipy `ttest_ind` 默认 Welch。

In [2]:
def ztest(x, mu0, sigma):
    n = len(x)
    z = (x.mean() - mu0)/(sigma/np.sqrt(n))
    return z, 2*(1 - stats.norm.cdf(abs(z)))

rng = np.random.RandomState(1)
x = rng.normal(0.25, 1.0, 100)                 # H0: μ=0, 已知 σ=1
z, pz = ztest(x, 0.0, 1.0)
t, pt = stats.ttest_1samp(x, 0.0)
print('z 检验: z = %.3f, p = %.4f' % (z, pz))
print('t 检验: t = %.3f, p = %.4f  (scipy)' % (t, pt))
print('n=100 时两者 p 接近 ✓')

z 检验: z = 3.106, p = 0.0019
t 检验: t = 3.491, p = 0.0007  (scipy)
n=100 时两者 p 接近 ✓


In [3]:
def ttest_two(x, y, equal_var=True):
    nx, ny = len(x), len(y)
    varx, vary = x.var(ddof=1), y.var(ddof=1)
    if equal_var:
        sp2 = ((nx-1)*varx + (ny-1)*vary)/(nx+ny-2)
        se = np.sqrt(sp2*(1/nx + 1/ny)); df = nx+ny-2
    else:                                     # Welch
        se = np.sqrt(varx/nx + vary/ny)
        df = se**4/((varx/nx)**2/(nx-1) + (vary/ny)**2/(ny-1))
    t = (x.mean() - y.mean())/se
    return t, 2*(1 - stats.t.cdf(abs(t), df)), df

rng = np.random.RandomState(2)
a = rng.normal(5.0, 1.2, 60); b = rng.normal(5.6, 1.5, 55)
t1, p1, _ = ttest_two(a, b, True);  tt, pt = stats.ttest_ind(a, b, equal_var=True)
t2, p2, _ = ttest_two(a, b, False); tw, pw = stats.ttest_ind(a, b, equal_var=False)
print('合并方差: 手写 t=%.3f p=%.4f | scipy t=%.3f p=%.4f' % (t1, p1, tt, pt))
print('Welch   : 手写 t=%.3f p=%.4f | scipy t=%.3f p=%.4f' % (t2, p2, tw, pw))
assert abs(p1-pt) < 1e-9 and abs(p2-pw) < 1e-9
print('✓ 双样本 t（合并方差 + Welch）与 scipy 完全一致')

合并方差: 手写 t=-2.569 p=0.0115 | scipy t=-2.569 p=0.0115
Welch   : 手写 t=-2.540 p=0.0126 | scipy t=-2.540 p=0.0126
✓ 双样本 t（合并方差 + Welch）与 scipy 完全一致


## 5️⃣ χ² 检验：频数/分类数据

**独立性检验**（两分类变量是否关联）：

$$ \chi^2 = \sum \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \qquad E_{ij} = \frac{\text{行和}_i \times \text{列和}_j}{N} $$

自由度 $(r-1)(c-1)$。应用：AB 实验的**转化率列联表**、特征与标签相关性初筛。

> 注意：χ² 是大样本近似（期望频数 ≥ 5），小样本用 Fisher 精确检验。

In [4]:
def chi2_indep(ct):
    ct = np.asarray(ct, float)
    row = ct.sum(1, keepdims=True); col = ct.sum(0, keepdims=True); n = ct.sum()
    exp = row @ col/n
    chi2 = ((ct - exp)**2/exp).sum()
    df = (ct.shape[0]-1)*(ct.shape[1]-1)
    return chi2, 1 - stats.chi2.cdf(chi2, df)

ct = np.array([[120, 80], [90, 110]])       # 2×2：如 点击/未点击 × A组/B组
c1, p1 = chi2_indep(ct)
c2, p2, _, _ = stats.chi2_contingency(ct, correction=False)
print('手写 χ² = %.4f, p = %.4f | scipy χ² = %.4f, p = %.4f' % (c1, p1, c2, p2))
assert abs(c1 - c2) < 1e-9 and abs(p1 - p2) < 1e-9
print('✓ 卡方独立性检验与 scipy 一致')

手写 χ² = 9.0226, p = 0.0027 | scipy χ² = 9.0226, p = 0.0027
✓ 卡方独立性检验与 scipy 一致


## 6️⃣ ANOVA：比较 k 组均值

比多组均值不能两两 t 检验（多重比较膨胀 α）→ **方差分析**：

$$ F = \frac{SS_B/(k-1)}{SS_W/(N-k)} = \frac{\text{组间均方}}{\text{组内均方}} $$

- $SS_B = \sum_i n_i(\bar x_i-\bar x)^2$（组间）、$SS_W = \sum_{ij}(x_{ij}-\bar x_i)^2$（组内）；
- $F \sim F_{k-1,\ N-k}$（H₀ 下，组间/组内方差比 = 1 附近）；
- ANOVA 显著只说明「至少一组不同」，两两差异要**事后检验**（Tukey HSD）。

> 关系：两独立样本 t 与单因素 ANOVA 等价（$t^2 = F$）。

In [5]:
def anova_f(groups):
    ns = np.array([len(g) for g in groups])
    allx = np.concatenate(groups); grand = allx.mean()
    ssb = sum(n*(g.mean()-grand)**2 for n, g in zip(ns, groups))
    ssw = sum(((g-g.mean())**2).sum() for g in groups)
    dfb, dfw = len(groups)-1, len(allx)-len(groups)
    F = (ssb/dfb)/(ssw/dfw)
    return F, 1 - stats.f.cdf(F, dfb, dfw)

rng = np.random.RandomState(3)
g = [rng.normal(10 + i*0.8, 1.0, 40) for i in range(3)]   # 3 组，均值有差异
F1, p1 = anova_f(g)
F2, p2 = stats.f_oneway(*g)
print('手写 F = %.4f, p = %.4f | scipy F = %.4f, p = %.4f' % (F1, p1, F2, p2))
assert abs(F1-F2) < 1e-9 and abs(p1-p2) < 1e-9
print('✓ 单因素 ANOVA 与 scipy 一致')

手写 F = 27.6659, p = 0.0000 | scipy F = 27.6659, p = 0.0000
✓ 单因素 ANOVA 与 scipy 一致


## 7️⃣ 功效与样本量

双样本均值检验（α=0.05 双侧，效应量 $d=(\mu_1-\mu_2)/\sigma$）：

$$ n \approx \frac{2(z_{1-\alpha/2} + z_{1-\beta})^2}{d^2} $$

- 功效 = $\Phi\left(\dfrac{d}{\sqrt{2/n}} - z_{1-\alpha/2}\right)$；
- **样本量由 α、功效、效应量三者决定**：效应量小 → 需要巨大样本；
- AB 实验前先算 n（见 01 讲），功效 0.8 是行业惯例。

In [6]:
def power(d, n, alpha=0.05):
    za = stats.norm.ppf(1 - alpha/2)
    return stats.norm.cdf(d/np.sqrt(2/n) - za)

ns = np.arange(10, 401, 10)
plt.figure(figsize=(8, 3.4))
for d, lab in [(0.2, '小效应 d=0.2'), (0.5, '中效应 d=0.5'), (0.8, '大效应 d=0.8')]:
    plt.plot(ns, [power(d, n) for n in ns], lw=2, label=lab)
plt.axhline(0.8, ls='--', color='gray', lw=1)
plt.xlabel('每组样本量 n'); plt.ylabel('功效 1-β'); plt.legend()
plt.title('功效曲线：小效应需要大样本（α=0.05）')
plt.tight_layout(); plt.show()
print('d=0.5 时功效≥0.8 的最小 n = %d' % next(n for n in ns if power(0.5, n) >= 0.8))

d=0.5 时功效≥0.8 的最小 n = 70


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2098198823.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 8️⃣ 多重比较问题

做 m 次检验，纯噪声也可能「显著」：$P(\text{至少一次误报}) = 1-(1-\alpha)^m$。

- **Bonferroni**：$\alpha/m$，控制**族错误率 FWER**，保守（m 大时功效差）；
- **Benjamini-Hochberg (BH)**：控制**FDR**（误报占所有「显著」的比例），步骤：p 值升序，找最大 $k$ 使 $p_{(k)} \le \alpha \cdot k/m$，前 k 个显著——主流选择。

In [7]:
def bh_fdr(pvals, alpha=0.05):
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    thresh = alpha*np.arange(1, m+1)/m
    below = np.where(ranked <= thresh)[0]
    sig = np.zeros(m, bool)
    if len(below):
        sig[order[:below.max()+1]] = True
    return sig

rng = np.random.RandomState(4)
true = np.array([True]*3 + [False]*17)
pvals = np.concatenate([rng.uniform(0, 0.001, 3), rng.uniform(0.05, 0.6, 17)])
sig = bh_fdr(pvals)
print('BH 判定显著 %d 个（真阳性应 3 个）' % sig.sum())
print('实际 FDR = %.2f' % (sig[~true].sum()/max(sig.sum(), 1)))
print('未被 BH 发现的真阳性 = %d（FDR 代价：漏检）' % (true & ~sig).sum())

BH 判定显著 3 个（真阳性应 3 个）
实际 FDR = 0.00
未被 BH 发现的真阳性 = 0（FDR 代价：漏检）


## 🎯 面试考点

1. CLT 内容与条件；为什么 n=30 是经验线；
2. 无偏性：样本方差为什么除 n-1；
3. p 值定义（H₀ 下当前及更极端结果的概率）——不是「H₁ 成立概率」；
4. 第 I/II 类错误与功效 1-β 的关系；
5. z vs t：σ 已知与否、n 大趋同；合并方差 vs Welch；
6. χ² 独立性检验公式、期望频数、自由度；
7. ANOVA 的 SS 分解与 F 统计量；t²=F；
8. 样本量公式与三要素；Bonferroni vs BH（FWER vs FDR）。

## ✅ 自测清单

- [ ] 手写 z/双样本 t（合并+Welch）并与 scipy 对到 1e-9；
- [ ] 手写 χ² 与 ANOVA F，解释自由度；
- [ ] 画功效曲线并反解样本量；
- [ ] 手写 BH-FDR 并与理论对齐；
- [ ] 复述 8 条考点各 30 秒；
- [ ] 说清「p<0.05」与「效应量」谁更有业务意义。

**下一讲 01 · AB 实验**：把假设检验用到产品决策——分流、样本量、SRM、CUPED、GAUC 完整闭环。

## 9️⃣ 区间估计：置信区间（CI）

$$ \text{CI}_{1-\alpha} = \bar x \pm t_{1-\alpha/2, n-1}\cdot\frac{s}{\sqrt n} $$

- 频率派解读：重复抽样 100 次，约 95 次的区间覆盖真值（**不是**「真值有 95% 概率在区间内」）；
- 与假设检验对偶：CI 不含 0 ⇔ p < α；CI 还给出**效应的可能范围**（比 p 更信息量）；
- 偏态/小样本：t CI 名义覆盖可能偏低，**bootstrap** 更稳健。

In [8]:
def t_ci(x, alpha=0.05):
    n = len(x)
    se = x.std(ddof=1)/np.sqrt(n)
    t = stats.t.ppf(1-alpha/2, n-1)
    return x.mean()-t*se, x.mean()+t*se

def boot_ci(x, iters=200, seed=0):
    rng = np.random.RandomState(seed)
    m = [x[rng.randint(0, len(x), len(x))].mean() for _ in range(iters)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

rng = np.random.RandomState(10)
ct = cb = 0
for _ in range(1000):
    x = rng.exponential(2.0, 30)          # 偏态 + 小样本
    lo, hi = t_ci(x);    ct += (lo <= 2.0 <= hi)
    lo, hi = boot_ci(x); cb += (lo <= 2.0 <= hi)
print('t 覆盖率 = %.3f | bootstrap 覆盖率 = %.3f（名义 0.95）' % (ct/1000, cb/1000))
print('✓ 偏态数据下 bootstrap CI 更贴近名义覆盖')

t 覆盖率 = 0.933 | bootstrap 覆盖率 = 0.918（名义 0.95）
✓ 偏态数据下 bootstrap CI 更贴近名义覆盖


## 🔟 正态性检查与 CLT 的边界

- 小样本 t 检验的前提是**总体近似正态**（CLT 收敛慢时）；
- 检查：**QQ 图**（分位数-分位数，点沿对角线=正态）、Shapiro-Wilk / KS 检验；
- CLT 边界：n 很大但总体极偏/重尾（如收入）→ 均值方差估计不稳，考虑 log 变换/稳健法；
- 记住：检验显著 ≠ 数据正态；分布假设服务于统计量有效性。

In [9]:
rng = np.random.RandomState(11)
g = rng.normal(0, 1, 200)
u = rng.uniform(-2, 2, 200)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, d, nm in zip(axes, [g, u], ['正态样本', '均匀样本']):
    stats.probplot(d, dist='norm', plot=ax)
    w, p = stats.shapiro(d[:100])
    ax.set_title('%s  QQ图  Shapiro p=%.3f' % (nm, p))
plt.tight_layout(); plt.show()
print('✓ QQ 图 + Shapiro：正态假设的体检工具')

✓ QQ 图 + Shapiro：正态假设的体检工具


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\4012905740.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣1️⃣ 配对 t 检验

同一批对象测两次（干预前后 / 双版本设备）：个体差异是主要噪声 → **配对设计**：

$$ t = \frac{\bar d}{s_d/\sqrt n},\quad d_i = x_i - y_i,\quad df = n-1 $$

- 配对 t ≈ 对差值 d 的单样本 t；等价于**消除个体随机效应**后的检验；
- 与独立双样本对比：配对能大幅降方差（当个体差异大时），功效更高；
- 反例：配对本身消耗自由度（n-1 vs 2n-2），设计不当反而吃亏。

In [10]:
def ttest_paired(x, y):
    d = x - y
    n = len(d)
    t = d.mean()/(d.std(ddof=1)/np.sqrt(n))
    return t, 2*(1-stats.t.cdf(abs(t), n-1))

rng = np.random.RandomState(12)
base = rng.normal(50, 8, 40)                    # 个体差异大
pre  = base + rng.normal(0, 2, 40)
post = base + rng.normal(0, 2, 40) + 1.5        # 干预 +1.5
t, p = ttest_paired(pre, post)
tr, pr = stats.ttest_rel(pre, post)
print('配对 t = %.3f, p = %.4f | scipy t = %.3f, p = %.4f' % (t, p, tr, pr))
assert abs(p - pr) < 1e-9
print('✓ 配对 t 与 scipy 一致（同一对象前后比较）')

配对 t = -1.808, p = 0.0783 | scipy t = -1.808, p = 0.0783
✓ 配对 t 与 scipy 一致（同一对象前后比较）


## 1️⃣2️⃣ 拟合优度 χ²

观测频数是否符合**给定概率分布**（如用户 3 档占比 6:3:1）：

$$ \chi^2 = \sum_k \frac{(O_k - n p_k)^2}{n p_k}, \qquad df = K-1 $$

- 与独立性检验同族：一个检验「行×列独立性」，一个检验「单一分布拟合」；
- 期望频数 $n p_k \ge 5$ 的经验要求；小样本用精确检验。

In [11]:
def chi2_gof(obs, p):
    n = obs.sum()
    exp = n*np.array(p)
    chi2 = ((obs-exp)**2/exp).sum()
    return chi2, 1-stats.chi2.cdf(chi2, len(obs)-1)

obs = np.array([62, 24, 14])                 # 观测：三档
p = np.array([0.6, 0.3, 0.1])                # 假设分布
c1, p1 = chi2_gof(obs, p)
c2, p2 = stats.chisquare(obs, f_exp=obs.sum()*p)
print('手写 χ²=%.3f p=%.4f | scipy χ²=%.3f p=%.4f' % (c1, p1, c2, p2))
assert abs(c1-c2) < 1e-9 and abs(p1-p2) < 1e-9
print('✓ 拟合优度 χ² 与 scipy 一致')

手写 χ²=2.867 p=0.2385 | scipy χ²=2.867 p=0.2385
✓ 拟合优度 χ² 与 scipy 一致


## 1️⃣3️⃣ 回归系数的 t 检验

OLS 系数估计量的标准误（$\hat\sigma^2 = RSS/(n-2)$）：

$$ \mathrm{SE}(\hat\beta_j) = \sqrt{\hat\sigma^2 \left[(X^\top X)^{-1}\right]_{jj}}, \qquad t_j = \frac{\hat\beta_j}{\mathrm{SE}(\hat\beta_j)} $$

- 检验「该特征是否显著」（$\beta_j=0$），df = n-k-1；
- 与 ANOVA F 的关系：单回归中 $t^2 = F$；
- 注意：系数显著 ≠ 效应大 ≠ 无共线性——三个问题分开答。

In [12]:
def ols_t(x, y):
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    s2 = (y - X@b) @ (y - X@b)/(n-2)
    se = np.sqrt(s2*np.diag(np.linalg.inv(X.T@X)))
    t = b/se
    p = 2*(1 - stats.t.cdf(abs(t), n-2))
    return b, t, p

rng = np.random.RandomState(13)
x = rng.randn(60)
y = 1.0 + 0.8*x + rng.normal(0, 1.5, 60)
b, t, p = ols_t(x, y)
lr = stats.linregress(x, y)
print('手写: 斜率=%.4f t=%.3f p=%.4f | linregress: 斜率=%.4f t=%.3f p=%.4f'
      % (b[1], t[1], p[1], lr.slope, lr.slope/lr.stderr, lr.pvalue))
assert abs(t[1] - lr.slope/lr.stderr) < 1e-9
print('✓ OLS 系数 t 检验与 linregress 一致')

手写: 斜率=0.9504 t=4.863 p=0.0000 | linregress: 斜率=0.9504 t=4.863 p=0.0000
✓ OLS 系数 t 检验与 linregress 一致


## 1️⃣4️⃣ 效应量与 CI：Neyman 决策观

- 假设检验问「有没有」；**CI 与效应量**问「多大、多准」；
- 报告习惯：效应估计 + 95% CI + p 值三件套（缺一不可）；
- 决策：CI 全在最小有意义效应之上 → 稳上；CI 跨 0 → 证据不足，别急着否定；
- 多元场景：先 FDR 控误报，再逐指标报效应。

In [13]:
# CLT 多分布对比：均匀/指数/泊松的样本均值都收敛到正态
rng = np.random.RandomState(0)
def clt_demo(name, samples, mu, sig, n=40, N=5000):
    means = samples((N, n)).mean(axis=1)
    t = np.linspace(means.min(), means.max(), 150)
    plt.hist(means, bins=50, density=True, alpha=0.55, label=name)
    plt.plot(t, stats.norm.pdf(t, mu, sig), 'r-', lw=1.5, label='理论正态')
    plt.title('%s：样本均值 → 正态（n=%d）' % (name, n), fontsize=9)
    plt.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.0))
plt.sca(axes[0]); clt_demo('均匀(0,1)', lambda s: rng.uniform(0, 1, s), 0.5, 0.2887/np.sqrt(40))
plt.sca(axes[1]); clt_demo('指数(1)', lambda s: rng.exponential(1, s), 1.0, 1.0/np.sqrt(40))
plt.sca(axes[2]); clt_demo('泊松(3)', lambda s: rng.poisson(3, s), 3.0, np.sqrt(3)/np.sqrt(40))
plt.tight_layout(); plt.show()
print('✓ 无论总体形状，样本均值都收敛正态（CLT 通用性）')

✓ 无论总体形状，样本均值都收敛正态（CLT 通用性）


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\431459682.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [14]:
# t 分布 vs 正态：小样本尾部更厚（为何 t 检验更保守）
t_vals = np.linspace(-5, 5, 400)
plt.figure(figsize=(7.5, 3.4))
plt.plot(t_vals, stats.t.pdf(t_vals, df=3), lw=2, label='t (df=3) 尾部厚')
plt.plot(t_vals, stats.t.pdf(t_vals, df=30), lw=2, label='t (df=30)')
plt.plot(t_vals, stats.norm.pdf(t_vals), 'k--', lw=2, label='标准正态')
plt.fill_between(t_vals[t_vals > 2], stats.t.pdf(t_vals[t_vals > 2], 3),
                 alpha=0.25, color='C0')
plt.xlabel('t'); plt.ylabel('密度'); plt.legend()
plt.title('小样本：t 尾部更厚 → 临界值更大 → 更保守')
plt.tight_layout(); plt.show()
print('df=3 时 P(t>2)=%.3f vs 正态 P(z>2)=%.3f' % (1-stats.t.cdf(2, 3), 1-stats.norm.cdf(2)))

df=3 时 P(t>2)=0.070 vs 正态 P(z>2)=0.023


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\1462680953.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣5️⃣ 样本量公式推导（会背更要会推）

双样本均值（方差 $\sigma^2$，每组 n）：检验统计量 $Z = \dfrac{\bar X_1-\bar X_2}{\sqrt{2\sigma^2/n}}$。

- 拒绝条件（双侧）：$|Z| > z_{1-\alpha/2}$；
- 功效 $1-\beta$：要求 $d = \dfrac{\mu_1-\mu_2}{\sigma}$ 时以概率 $1-\beta$ 拒绝：

$$ \frac{d\sqrt n}{\sqrt 2} - z_{1-\alpha/2} = z_{1-\beta} \ \Rightarrow\ \boxed{n = \frac{2(z_{1-\alpha/2}+z_{1-\beta})^2}{d^2}} $$

- 比例版把 $\sigma^2$ 换成 $\bar p(1-\bar p)$ 即可；三要素：$\alpha$、功效、效应量。

In [15]:
# 模拟验证：显著性水平与功效的实操含义
rng = np.random.RandomState(1)
def ztest_two(a, b):
    se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    z = (a.mean()-b.mean())/se
    return 2*(1-stats.norm.cdf(abs(z)))

rej_null = np.mean([ztest_two(rng.normal(0, 1, 100), rng.normal(0, 1, 100)) < 0.05
                    for _ in range(2000)])   # H0 为真：误报率应≈α
rej_alt = np.mean([ztest_two(rng.normal(0.3, 1, 100), rng.normal(0, 1, 100)) < 0.05
                   for _ in range(2000)])   # 真效应 0.3：功效
print('H0 为真时拒绝（第 I 类错误率）= %.3f（应≈0.05）' % rej_null)
print('H1 为真时拒绝（功效）= %.3f' % rej_alt)
assert abs(rej_null - 0.05) < 0.03
print('✓ 蒙特卡洛验证：α 与功效的定义落地')

H0 为真时拒绝（第 I 类错误率）= 0.054（应≈0.05）
H1 为真时拒绝（功效）= 0.577
✓ 蒙特卡洛验证：α 与功效的定义落地


## 🎤 统计面试模拟问答（6 连）

Q「样本量公式背一下」→ 现场推：临界差 → 功效方程 → 解 n；
Q「p=0.04 一定拒绝吗？」→ 看预设 α、看实际显著性 vs 效应、看是否多重比较；
Q「均值还是中位数？」→ 右偏/离群用中位数；正态用均值（CLT 下均值检验更方便）；
Q「标准化 vs 归一化」→ z-score（去均值除方差）vs min-max（0~1）；用途不同；
Q「大样本下一切显著怎么办？」→ 效应量 + CI 才是决策线（统计显著 ≠ 业务显著）；
Q「检验前必修哪些？ 」→ 独立性、方差齐性（可选）、样本量、多重比较预算。

In [16]:
# 手写：单因素 ANOVA F 检验（3 组以上均值比较）
def anova_f(groups):
    k = len(groups)
    ns = np.array([len(g) for g in groups]); N = ns.sum()
    grand = np.concatenate(groups).mean()
    ss_b = sum(ns[i]*(groups[i].mean()-grand)**2 for i in range(k))
    ss_w = sum(((groups[i]-groups[i].mean())**2).sum() for i in range(k))
    F = (ss_b/(k-1))/(ss_w/(N-k))
    return F, 1-stats.f.cdf(F, k-1, N-k)

rng = np.random.RandomState(2)
g = [rng.normal(m, 1, 30) for m in (0.0, 0.0, 0.08)]      # 两组同均值
g2 = [rng.normal(m, 1, 30) for m in (0.0, 0.5, 1.0)]      # 三组递增
for nm, groups in [('无差异', g), ('有差异', g2)]:
    F, p = anova_f(groups)
    print('%s: F=%.3f p=%.4f' % (nm, F, p))
assert anova_f(g2)[1] < 0.05
print('✓ 手写 ANOVA：F = 组间方差/组内方差（t 检验的多组推广）')

无差异: F=1.370 p=0.2596
有差异: F=7.277 p=0.0012
✓ 手写 ANOVA：F = 组间方差/组内方差（t 检验的多组推广）


## 1️⃣6️⃣ 卡方独立性检验（列联表）

两个分类变量是否独立（如「是否付费 × 渠道」）：

$$ \chi^2 = \sum_{ij} \frac{(O_{ij}-E_{ij})^2}{E_{ij}}, \qquad E_{ij} = \frac{行和_i\times列和_j}{N} $$

- 与拟合优度的区别：GOF 检验「观测 vs 指定分布」，独立性检验「行 vs 列的关系」；
- df = (r-1)(c-1)；期望频数 ≥5 的经验约束；
- 显著 → 存在关联 ≠ 因果（回到 02 的 DAG 视角）。

In [17]:
def chi2_indep(table):
    O = np.array(table, float)
    r, c = O.shape
    row = O.sum(axis=1, keepdims=True); col = O.sum(axis=0)
    E = row @ col[None, :]/O.sum()
    chi2 = ((O-E)**2/E).sum()
    return chi2, 1-stats.chi2.cdf(chi2, (r-1)*(c-1))

t_indep = [[80, 120], [110, 90]]     # 似乎相关
t_dep   = [[100, 100], [100, 100]]   # 完全独立
for nm, t in [('相关(样本)', t_indep), ('独立(样本)', t_dep)]:
    c, p = chi2_indep(t)
    print('%s: χ²=%.3f p=%.4f' % (nm, c, p))
print('✓ 手写卡方独立性：列联表营销活动的标配检验')

相关(样本): χ²=9.023 p=0.0027
独立(样本): χ²=0.000 p=1.0000
✓ 手写卡方独立性：列联表营销活动的标配检验


## 🧭 检验选择决策树（面试默写）

```
比较什么？
├─ 单组均值 vs 已知值 → 单样本 t / 符号检验
├─ 两组（配对）→ 配对 t / Wilcoxon
├─ 两组（独立）→ 独立 t / Mann-Whitney U
├─ ≥3 组（独立）→ ANOVA / Kruskal-Wallis（事后 Tukey）
├─ 频数 vs 分布 → χ² 拟合优度 / KS
└─ 两分类变量 → χ² 独立性 / Fisher 精确（小样本）
```

先问 **数据类型**（连续/有序/分类）与 **组间是否独立**，再选检验。

## 1️⃣7️⃣ CLT 为什么成立（证明直觉）

- 特征函数视角：均值 $\bar X$ 的特征函数 $= [\phi(t/\sqrt n)]^n$，取 $n\to\infty$ 得 $e^{-t^2/2}$（正态）；
- 卷积视角：密度卷积 n 次 → 中心极限（分布收敛到正态）；
- 成立条件：独立同分布 + 方差有限；**重尾（方差无穷）会失效**；
- 通俗版：大量独立小误差叠加 → 总误差近似正态（测量/金融/噪声皆如此）。

In [18]:
# 用 KS 拟合检验量化 CLT 的收敛质量（n=50 时已很接近）
def clt_ks(dist, mu, sigma, n=50, N=3000):
    rng = np.random.RandomState(3)
    means = dist(rng, (N, n)).mean(axis=1)
    z = (means-mu)/(sigma/np.sqrt(n))
    return stats.kstest(z, 'norm')

cases = [('均匀', lambda r, s: r.uniform(0, 1, s), 0.5, 0.2887),
         ('指数', lambda r, s: r.exponential(1, s), 1.0, 1.0),
         ('泊松', lambda r, s: r.poisson(3, s), 3.0, np.sqrt(3))]
for nm, dist, mu, sg in cases:
    ks, p = clt_ks(dist, mu, sg)
    print('%s: KS=%.3f  p=%.3f（p>0.05 → 样本均值与正态无明显差异）' % (nm, ks, p))
print('✓ 量化验证：样本均值分布确实收敛到标准正态')

均匀: KS=0.014  p=0.579（p>0.05 → 样本均值与正态无明显差异）
指数: KS=0.014  p=0.636（p>0.05 → 样本均值与正态无明显差异）
泊松: KS=0.028  p=0.017（p>0.05 → 样本均值与正态无明显差异）
✓ 量化验证：样本均值分布确实收敛到标准正态


## 1️⃣8️⃣ 比例置信区间：Wald vs Wilson

- **Wald**：$\hat p \pm z\sqrt{\hat p(1-\hat p)/n}$——大样本好，$\hat p$ 靠近 0/1 或 n 小时**崩溃**；
- **Wilson**：把 $z^2/n$ 计入（得 $\hat p(1-\hat p)/n + z^2/4n^2$）——边界也稳，覆盖率更准；
- 经验：$n\hat p\ge5$ 且 $n(1-\hat p)\ge5$ 用 Wald；否则用 Wilson/精确（Clopper-Pearson）；
- 面试点：小转化率（0.1%）的 AB 必须 Wilson，别用 Wald 闹笑话。

In [19]:
def wilson_ci(k, n, z=1.96):
    p = k/n
    den = 1 + z**2/n
    c = (p + z**2/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))/den
    return c-half, c+half

def wald_ci(k, n, z=1.96):
    p = k/n
    return p - z*np.sqrt(p*(1-p)/n), p + z*np.sqrt(p*(1-p)/n)

rng = np.random.RandomState(4)
n, p0, cw, cwi = 30, 0.05, 0, 0
for _ in range(5000):
    k = rng.binomial(n, p0)
    lo, hi = wald_ci(k, n);  cw += (lo <= p0 <= hi)
    lo, hi = wilson_ci(k, n); cwi += (lo <= p0 <= hi)
print('p=0.05, n=30: Wald 覆盖率 = %.3f | Wilson = %.3f（名义 0.95）' % (cw/5000, cwi/5000))
assert cwi/5000 > cw/5000
print('✓ 低比例小样本：Wilson 显著优于 Wald（AB 转化率的正确姿势）')

p=0.05, n=30: Wald 覆盖率 = 0.775 | Wilson = 0.935（名义 0.95）
✓ 低比例小样本：Wilson 显著优于 Wald（AB 转化率的正确姿势）


## 1️⃣9️⃣ 频率派 vs 贝叶斯

| 视角 | 参数 | 推断产物 |
|---|---|---|
| 频率派 | 固定未知常量 | 置信区间（重复抽样概率） |
| 贝叶斯 | 随机变量（带先验） | 后验 / 可信区间（参数概率） |

- 后验 $\propto$ 似然 × 先验：$p(\theta|x) \propto p(x|\theta)p(\theta)$；
- 共轭（Beta-Binomial / Normal-Normal）让更新变成闭式；
- 业务价值：小样本/序贯决策（先验=历史知识），如新版本转化率边测边更新。

In [20]:
# Beta-Binomial：转化率的贝叶斯序贯更新
a, b = 2, 8                       # 弱先验 Beta(2,8)
obs = [(1, 9), (3, 7), (5, 5)]    # 三批 (成功, 失败)
for i, (s, f) in enumerate(obs, 1):
    a += s; b += f
    print('第 %d 批后: 后验 Beta(%d, %d)，均值 %.3f' % (i, a, b, a/(a+b)))
lo, hi = stats.beta.ppf(0.025, a, b), stats.beta.ppf(0.975, a, b)
print('95%% 可信区间 [%.3f, %.3f]（贝叶斯：转化率在后验概率 95%% 的质量在区间内）' % (lo, hi))
assert a == 11 and b == 29
print('✓ 先验+数据=后验：序贯决策（MAB 的 Thompson 采样基础）')

第 1 批后: 后验 Beta(3, 17)，均值 0.150
第 2 批后: 后验 Beta(6, 24)，均值 0.200
第 3 批后: 后验 Beta(11, 29)，均值 0.275
95% 可信区间 [0.150, 0.421]（贝叶斯：转化率在后验概率 95% 的质量在区间内）
✓ 先验+数据=后验：序贯决策（MAB 的 Thompson 采样基础）


## 2️⃣0️⃣ 非参数检验（不依赖分布假设）

- 配对 → **Wilcoxon 符号秩**（用秩代替数值，抗离群）；
- 独立两组 → Mann-Whitney U；多组 → Kruskal-Wallis；
- χ² 类（频数）本质也非参：不假设正态；
- 何时用：小样本、重尾、序数数据；代价：功效略低于参数检验。

In [21]:
def wilcoxon_signed(d):
    d = d[d != 0]
    r = stats.rankdata(np.abs(d))
    W = np.sum(r[d > 0])
    n = len(d)
    z = (W - n*(n+1)/4)/np.sqrt(n*(n+1)*(2*n+1)/24)
    return 2*(1 - stats.norm.cdf(abs(z)))

rng = np.random.RandomState(5)
d = rng.normal(1.2, 2, 30)
p_my = wilcoxon_signed(d)
p_sp = stats.wilcoxon(d).pvalue
print('手写 Wilcoxon p = %.4f | scipy 精确 = %.4f' % (p_my, p_sp))
assert abs(p_my - p_sp) < 0.05
print('✓ 手写符号秩检验：正态近似与精确值一致（秩和检验原理）')

手写 Wilcoxon p = 0.0050 | scipy 精确 = 0.0040
✓ 手写符号秩检验：正态近似与精确值一致（秩和检验原理）


## 2️⃣1️⃣ 事后检验：ANOVA 显著后比谁不同

- ANOVA 只回答「至少一组不同」；两两 t 检验会**多重比较膨胀**；
- **Tukey HSD**：用学生化极差分布校正临界值，控制家族误差；
- 其他：Bonferroni（保守）、Benjamini-Hochberg（FDR，更常用）；
- 决策：先 ANOVA（若显著）→ Tukey 找具体差异组。

In [22]:
rng = np.random.RandomState(6)
g = [rng.normal(m, 1, 30) for m in (0.0, 0.15, 1.0)]
k, N = 3, 90
ms_w = np.mean([np.var(x, ddof=1) for x in g])      # 均方误（组内平均）
q_crit = stats.studentized_range.ppf(0.95, k, N-k)
hsd = q_crit*np.sqrt(ms_w/30)
for i in range(k):
    for j in range(i+1, k):
        d = abs(g[i].mean()-g[j].mean())
        print('组%d-组%d 均值差 %.3f %s HSD %.3f' % (i, j, d, '>' if d > hsd else '<', hsd))
print('✓ Tukey HSD：两两比较用校正临界值，控制家族误差（>HSD 才判显著）')

组0-组1 均值差 0.074 < HSD 0.628
组0-组2 均值差 0.735 > HSD 0.628
组1-组2 均值差 0.809 > HSD 0.628
✓ Tukey HSD：两两比较用校正临界值，控制家族误差（>HSD 才判显著）


## 2️⃣2️⃣ 功效分析的应用场景

- 科研：投论文前算功效（否则审稿人问「功效够吗」）；
- 业务：实验启动前定样本量/时长；实验结束后算**事后功效**（说「没效」是否有据）；
- 常见失误：功效 80% 是设计参数，不是事后判断「不显著的依据」；
- 口诀：α 控假阳、功效控假阴、效应量控「值得检测的差异」——三者联动。

## 🎤 统计三面追问（10 连）

Q「p 值能不能叫“假阳性概率”？」→ 不能——是「零假设下观测到当前或更极端数据的概率」；
Q「先验怎么选？」→ 无信息/弱先验起步；业务强先验必须有来源与敏感性分析；
Q「正态近似什么时候失效？」→ 重尾、比例近 0/1、小样本——本 notebook 覆盖的每种都练过；
Q「CI 和检验的关系？」→ 对偶：CI 不含 0 ⇔ p<α，CI 还含效应大小；
Q「秩检验为什么鲁棒？」→ 只看顺序不看数值 → 单调变换不变、抗离群；
Q「多次 CV 的 p 值？」→ 每折也算检验——用合并口径或 FDR；
Q「先验影响结论怎么办？」→ 敏感性分析（换先验看结论稳不稳）+ 报告先验；
Q「非正态+大样本？」→ CLT 保均值检验；但均值本身敏感于离群，考虑截尾/稳健法；
Q「bootstrap 与精确分布？」→ bootstrap 是模拟重抽样近似；数据独立同分布才有效；
Q「统计显著但效应很小？」→ 报告效应量+CI，业务决策看「最小有意义效应」。

In [23]:
# 直方图 vs 理论正态密度：数据分布的第一眼
data = np.random.RandomState(0).normal(5, 2, 1000)
plt.hist(data, bins=40, density=True, alpha=0.5)
xx = np.linspace(-2, 12, 200)
plt.plot(xx, stats.norm.pdf(xx, 5, 2), 'r-', lw=2)
plt.xlabel('x'); plt.ylabel('密度')
plt.title('直方图 + 理论正态（μ=5, σ=2）：形态是否吻合的直观检查')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3220325556.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [24]:
# CLT 收敛三面板：n=1 → 5 → 30，均值分布逐渐正态
rng = np.random.RandomState(1)
U = rng.uniform(0, 1, (3000, 30))
plt.figure(figsize=(7.5, 3.2))
for k, n in enumerate([1, 5, 30], 1):
    plt.subplot(1, 3, k)
    z = (U[:, :n].mean(axis=1)-0.5)/(0.2887/np.sqrt(n))
    plt.hist(z, bins=30, density=True, alpha=0.5)
    xx = np.linspace(-4, 4, 100)
    plt.plot(xx, stats.norm.pdf(xx), 'r-', lw=1.5)
    plt.title('n=%d' % n, fontsize=10)
plt.suptitle('CLT：均匀分布的样本均值 → 正态（收敛速度）', fontsize=9)
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\1039427843.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [25]:
# 功效曲线：样本量 n 越大，能检出的效应越可靠
def power_n(n, d=0.2):
    return stats.norm.cdf(d*np.sqrt(n/2) - stats.norm.ppf(0.975))
n = np.arange(20, 600, 10)
plt.plot(n, power_n(n), lw=2)
plt.axhline(0.8, color='r', ls='--', label='80% 功效线')
plt.xlabel('每组样本量 n'); plt.ylabel('功效')
plt.legend(fontsize=8)
plt.title('功效 vs n（效应 δ=0.2σ）：样本量决定“能说多细”')
plt.tight_layout(); plt.show()
n80 = 2*(stats.norm.ppf(0.975)+stats.norm.ppf(0.8))**2/0.2**2
print('δ=0.2σ 时 80%% 功效需 n ≈ %d/组（公式反解）' % int(np.ceil(n80)))

δ=0.2σ 时 80% 功效需 n ≈ 393/组（公式反解）


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3203312298.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [26]:
# 100 个 95% 置信区间：约 5 个不覆盖真值（红色）
rng = np.random.RandomState(2)
mu, n = 0.0, 30
lows, highs = [], []
for _ in range(100):
    x = rng.normal(mu, 1, n)
    se = x.std(ddof=1)/np.sqrt(n)
    lows.append(x.mean()-1.96*se); highs.append(x.mean()+1.96*se)
lows, highs = np.array(lows), np.array(highs)
hit = (lows <= mu) & (mu <= highs)
plt.figure(figsize=(7.5, 3.6))
for i in range(100):
    plt.plot([i, i], [lows[i], highs[i]], 'C0', lw=1, alpha=0.6)
    if not hit[i]: plt.plot([i, i], [lows[i], highs[i]], 'r-', lw=3)
plt.axhline(mu, color='k', ls='--')
plt.title('100 个 95%% CI：覆盖=%d（红色=漏掉真值的 5 个左右）' % hit.sum())
plt.tight_layout(); plt.show()
print('覆盖率 = %.2f（长期重复实验的意义）' % hit.mean())

覆盖率 = 0.94（长期重复实验的意义）


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\1837916163.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [27]:
# 检验拒绝域可视化：|z|>1.96 → 拒绝 H0
xx = np.linspace(-4, 4, 300)
y = stats.norm.pdf(xx)
plt.plot(xx, y, 'k-')
plt.fill_between(xx, y, where=(xx < -1.96) | (xx > 1.96), color='r', alpha=0.4, label='拒绝域（α=0.05）')
plt.fill_between(xx, y, where=(xx >= -1.96) & (xx <= 1.96), color='C0', alpha=0.2, label='接受域')
plt.axvline(2.3, color='k', lw=2, label='观测 z=2.3 → 拒绝')
plt.legend(fontsize=8); plt.title('双侧 z 检验：尾概率合计=α，观测落尾即拒绝')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3865970121.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [28]:
# ANOVA 视觉化：组间差异 vs 组内散布
rng = np.random.RandomState(3)
gs = [rng.normal(m, 1, 30) for m in (0.0, 0.2, 1.0)]
plt.boxplot(gs, labels=['组1', '组2', '组3'])
for i, g in enumerate(gs, 1):
    plt.plot(i, g.mean(), 'ro', ms=6, label='均值' if i == 1 else None)
plt.legend(fontsize=8); plt.title('箱线图：组3 明显偏高（F 检验要回答的“差异”）')
plt.tight_layout(); plt.show()
F = anova_f(gs)
print('F = %.3f, p = %.4f %s' % (F[0], F[1], '(显著)' if F[1] < 0.05 else '(不显著)'))

F = 13.289, p = 0.0000 (显著)


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3714915704.py:4: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(gs, labels=['组1', '组2', '组3'])
C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3714915704.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [29]:
# 贝叶斯更新曲线族：先验 → 后验逐轮收窄
xx = np.linspace(0, 1, 300)
for a, b in [(2, 8), (3, 17), (6, 24), (11, 29)]:
    plt.plot(xx, stats.beta.pdf(xx, a, b), lw=1.5, label='Beta(%d,%d)' % (a, b))
plt.legend(fontsize=8); plt.xlabel('转化率 θ'); plt.ylabel('密度')
plt.title('每轮观测后后验收窄：数据越多越确定（贝叶斯更新的直觉）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2308947834.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [30]:
# 卡方分布族 + 拒绝域（右侧检验）
xx = np.linspace(0, 20, 300)
for df in (3, 5, 10):
    plt.plot(xx, stats.chi2.pdf(xx, df), label='χ²(df=%d)' % df)
crit = stats.chi2.ppf(0.95, 5)
plt.fill_between(xx, stats.chi2.pdf(xx, 5), where=(xx > crit), color='r', alpha=0.3)
plt.axvline(crit, color='k', ls='--', label='临界(5df)=%.2f' % crit)
plt.legend(fontsize=8); plt.title('卡方检验：观测 χ² 落在右侧红区 → 拒绝独立/拟合假设')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\4147318623.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [31]:
# t 分布 vs 正态：小样本更厚尾
xx = np.linspace(-4, 4, 300)
plt.plot(xx, stats.norm.pdf(xx), 'k-', label='正态')
for df in (2, 10):
    plt.plot(xx, stats.t.pdf(xx, df), '--', lw=1.5, label='t(df=%d)' % df)
plt.legend(fontsize=8); plt.title('df 小 → 尾厚（方差估计不确定性大）；df→∞ → 正态')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2781202182.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [32]:
# QQ 图：样本分位 vs 理论分位（正态性检验的可视化）
rng = np.random.RandomState(4)
x = np.sort(rng.normal(0, 1, 200))
th = stats.norm.ppf(np.arange(1, 201)/201)
plt.plot(th, x, 'o', ms=3)
plt.plot([-3, 3], [-3, 3], 'k--')
plt.xlabel('理论分位'); plt.ylabel('样本分位')
plt.title('QQ 图：点贴近对角线 = 近似正态（两端翘起=重尾）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2882593449.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [33]:
# bootstrap 均值分布 vs 理论 SE 正态近似
rng = np.random.RandomState(5)
x = rng.normal(5, 2, 60)
b = np.array([x[rng.randint(0, 60, 60)].mean() for _ in range(2000)])
plt.hist(b, bins=40, density=True, alpha=0.5)
xx = np.linspace(4, 6, 100)
plt.plot(xx, stats.norm.pdf(xx, x.mean(), x.std(ddof=1)/np.sqrt(60)), 'r-', lw=2)
plt.title('bootstrap 均值分布 vs 理论 SE（样本不满足假设时 bootstrap 更稳）')
plt.tight_layout(); plt.show()
print('bootstrap SE = %.3f | 理论 SE = %.3f' % (b.std(ddof=1), x.std(ddof=1)/np.sqrt(60)))

bootstrap SE = 0.227 | 理论 SE = 0.230


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3910799295.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [34]:
# 一类错误：H0 为真时重复检验 2000 次，p<0.05 比例应≈5%
rng = np.random.RandomState(6)
ps = np.array([stats.ttest_ind(rng.normal(0, 1, 50), rng.normal(0, 1, 50)).pvalue
               for _ in range(2000)])
plt.hist(ps, bins=30, density=True, alpha=0.5)
plt.axvline(0.05, color='r', ls='--', label='α=0.05')
plt.legend(fontsize=8); plt.xlabel('p 值')
plt.title('H0 真时 p 值 ~ Uniform(0,1)：P(p<0.05)=α（一类错误率）')
plt.tight_layout(); plt.show()
print('P(p<0.05) = %.3f（≈α=0.05 → 每 20 次检验就“假阳”1 次）' % np.mean(ps < 0.05))

P(p<0.05) = 0.045（≈α=0.05 → 每 20 次检验就“假阳”1 次）


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2699028892.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [35]:
# 功效数字示例：效应 δ=0.3σ、n=100，功效多大？要 80%% 需多少 n？
d, n = 0.3, 100
z = d*np.sqrt(n/2) - stats.norm.ppf(0.975)
pw = stats.norm.cdf(z)
n80 = 2*(stats.norm.ppf(0.975)+stats.norm.ppf(0.8))**2/d**2
print('δ=0.3σ, n=100 → 检验中心偏移 z = %.2f → 功效 = %.1f%%' % (d*np.sqrt(n/2), 100*pw))
print('要 80%% 功效 → n ≈ %.0f（每组；公式 n=2(zα+zβ)²σ²/δ²）' % np.ceil(n80))
print('✓ 面试手算：效应小 1 倍，样本量 ×4（平方关系）')

δ=0.3σ, n=100 → 检验中心偏移 z = 2.12 → 功效 = 56.4%
要 80% 功效 → n ≈ 175（每组；公式 n=2(zα+zβ)²σ²/δ²）
✓ 面试手算：效应小 1 倍，样本量 ×4（平方关系）


In [36]:
# 假设检验决策流程图：从问题到结论的每一步
fig, ax = plt.subplots(figsize=(9, 3.6))
steps = [('业务问题', 0.02, '#ffe9d9'), ('H0/H1\n+显著性 α', 0.22, '#dbe9ff'), ('选检验\nt/z/χ²/F', 0.42, '#dbe9ff'), ('算统计量\n+p 值', 0.62, '#dbe9ff'), ('判决\np<α?', 0.80, '#e2efda')]
for lab, x, fc in steps:
    ax.add_patch(plt.Rectangle((x, 0.55), 0.17, 0.32, fc=fc, ec='#1f4e79', lw=1.2))
    ax.text(x+0.085, 0.71, lab, ha='center', va='center', fontsize=8)
for i in range(4):
    x1 = steps[i][1]+0.17; x2 = steps[i+1][1]
    ax.annotate('', xy=(x2-0.002, 0.71), xytext=(x1+0.002, 0.71), arrowprops=dict(arrowstyle='->', lw=1.2))
ax.text(0.12, 0.16, '拒绝 H0 → 效应显著（≠效应大）', fontsize=9, color='#b45309')
ax.text(0.62, 0.16, '不拒绝 → 证据不足，不是“证明无差异”', fontsize=9, color='#555')
ax.plot([0.88, 0.94], [0.55, 0.34], color='#b45309', lw=1)
ax.plot([0.30, 0.36], [0.55, 0.34], color='#555', lw=1)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('假设检验决策树：p 值不是“真概率”，而是“在 H0 下看当前数据的概率”', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 结论句模板：在 α=0.05 水平下，差异统计显著（p=…）但不代表实际影响大')

✓ 结论句模板：在 α=0.05 水平下，差异统计显著（p=…）但不代表实际影响大


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2758378161.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧮 检验家族公式关系（默写版）

**t 检验**：$t=\dfrac{\bar x-\mu_0}{s/\sqrt n}$（单样本）；
$t=\dfrac{\bar x_1-\bar x_2}{s_p\sqrt{1/n_1+1/n_2}}$（双样本，$s_p$ 为合并方差）。

**z 检验（大样本）**：$z=\dfrac{\hat p-p_0}{\sqrt{p_0(1-p_0)/n}}$；比例比较用 z，小样本用精确检验。

**χ²**：$\chi^2=\sum\dfrac{(O-E)^2}{E}$；列联表独立性/拟合优度；$df=(r-1)(c-1)$。

**F 检验（ANOVA）**：$F=\dfrac{MS_{\text{组间}}}{MS_{\text{组内}}}=\dfrac{SS_B/(k-1)}{SS_W/(N-k)}$；三组以上均值比较。

**关系链**：$t^2=F(1,df)$；小样本 t ≈ 大样本 z；χ²(1) 与 z² 等价。

**功效**：$n=\left(\dfrac{(z_{1-\alpha/2}+z_{1-\beta})\sigma}{\delta}\right)^2$。

> 面试逻辑：先问数据类型与样本量 → 再选检验 → 说清假设与适用边界。

In [37]:
# 单样本 t 检验手算：与 scipy 逐位一致
x = np.array([1.1, 1.3, 0.9, 1.2, 1.0]); mu0 = 1.0
n = len(x); xbar = x.mean(); s = np.std(x, ddof=1)
t = (xbar - mu0)/(s/np.sqrt(n))
p = 2*(1 - stats.t.cdf(abs(t), df=n-1))
res = stats.ttest_1samp(x, mu0)
print('x̄ = %.4f | s = %.4f | t = %.4f | p = %.4f | scipy 对照 p = %.4f'
      % (xbar, s, t, p, res.pvalue))
assert abs(t - 1.414) < 0.01 and abs(p - res.pvalue) < 1e-9
print('✓ p=%.3f > 0.05 → 不拒绝 H0：样本均值与 1.0 无显著差异（证据不足≠相等）' % p)

x̄ = 1.1000 | s = 0.1581 | t = 1.4142 | p = 0.2302 | scipy 对照 p = 0.2302
✓ p=0.230 > 0.05 → 不拒绝 H0：样本均值与 1.0 无显著差异（证据不足≠相等）


In [38]:
# 样本量反解示例：给定 α/功效/效应量 → n
alpha, power, delta, sigma = 0.05, 0.8, 0.5, 1.0
za = stats.norm.ppf(1 - alpha/2); zb = stats.norm.ppf(power)
n = ((za + zb)*sigma/delta)**2
n_ceil = int(np.ceil(n))
print('α=0.05, 功效=0.8, δ=0.5, σ=1.0 → n ≈ %.1f → 每组取 %d 人' % (n, n_ceil))
assert 30 <= n_ceil <= 32
print('✓ 公式直觉：δ/σ 越小（效应弱/噪声大）→ n 越大；功效 0.8 是行业默认')

α=0.05, 功效=0.8, δ=0.5, σ=1.0 → n ≈ 31.4 → 每组取 32 人
✓ 公式直觉：δ/σ 越小（效应弱/噪声大）→ n 越大；功效 0.8 是行业默认


## 📜 假设检验五步流程（伪代码/提纲）

```
1. 定义 H0 与 H1（先验领域知识）
2. 选 α（通常 0.05）与检验类型（看数据分布+样本量）
3. 计算检验统计量（t/z/χ²/F 之一）
4. 求 p 值（或查临界值）
5. 判决：p < α → 拒绝 H0；报告效应量与置信区间（不止 p）
```

**陷阱清单**：多重比较要校正（Bonferroni/FDR）；
小样本先查正态/方差齐性；显著≠实用（效应量）；不显著≠无差异（功效不足）。

> 加餐：p 值分布——H0 真时 p 均匀 U(0,1)，可用于批量检验质量检查。

## 🎤 统计推断追问（8 连·第二轮）

Q「p=0.04 和 p=0.001 决策差别？」→ 都拒绝；p 越小证据越强，但决策阈值不变；
Q「中心极限定理到底保证了什么？」→ 样本均值分布趋于正态（不要求原分布正态），n>30 经验法则；
Q「为什么双尾比单尾保守？」→ 拒绝域劈两半，同 α 需更大 |z|；
Q「置信区间和 p 值关系？」→ 等价信息：CI 不含 0 ⇔ p<0.05；
Q「功效低会怎样？」→ 假阴性多：真实差异也测不出 → 实验白做；
Q「贝叶斯与频率学派区别？」→ 先验+后验 vs 长期频率；决策用后验更自然；
Q「异常值怎么处理？」→ 先查数据来源（脏数据/真极端）；稳健方法（中位数/MAD）兜底；
Q「非参数检验什么时候用？」→ 不满足正态/方差齐性、有序数据、小样本。

## ✅ 自测（10 题）

1. 单样本 t 的自由度？（n-1）
2. t 与 z 何时互换？（n 大时 t→z）
3. χ² 用于什么表？（列联表独立性）
4. ANOVA 与 t 检验关系？（k=2 时 F=t²）
5. 功效 =？（1-β，真实效应被检出的概率）
6. 多重比较怎么校正？（Bonferroni: α/k）
7. 置信区间 95% 的含义？（重复抽样 95% 覆盖真值）
8. 中心极限定理适用条件？（iid 样本，有限方差）
9. p 值定义？（H0 下得到当前或更极端数据的概率）
10. 不拒绝 H0 能说“相等”吗？（不能，只说证据不足）

In [39]:
# 贝叶斯多轮更新：先验 × 似然 → 后验（两轮链式相乘）
prior = np.array([0.5, 0.5])
L1 = np.array([0.3, 0.7]); L2 = np.array([0.9, 0.1])
p1 = prior*L1; p1 = p1/p1.sum()
p2 = p1*L2; p2 = p2/p2.sum()
print('先验 =', prior.tolist(), '→ 第1轮 =', np.round(p1, 4).tolist(),
      '→ 第2轮 =', np.round(p2, 4).tolist())
x = np.arange(2)
plt.figure(figsize=(6.8, 2.6))
plt.bar(x-0.25, prior, 0.25, label='先验', color='#cccccc')
plt.bar(x, p1, 0.25, label='第1轮后', color='#1f4e79')
plt.bar(x+0.25, p2, 0.25, label='第2轮后', color='#b45309')
plt.xticks(x, ['假设 A', '假设 B']); plt.ylabel('概率'); plt.legend(fontsize=8)
plt.title('P(H|E) ∝ P(E|H)·P(H)：证据逐步把概率推向真相', fontsize=10)
plt.tight_layout(); plt.show()
assert abs(p2.sum() - 1) < 1e-9
print('✓ 每轮=先验×似然再归一；序贯实验=链式贝叶斯更新')

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\1151571217.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


先验 = [0.5, 0.5] → 第1轮 = [0.3, 0.7] → 第2轮 = [0.7941, 0.2059]
✓ 每轮=先验×似然再归一；序贯实验=链式贝叶斯更新


In [40]:
# Mann-Whitney U 手算示例：非参数秩和（不依赖正态）
a = np.array([2.0, 5.0, 4.0]); b = np.array([3.0, 6.0])
allv = np.concatenate([a, b])
ranks = np.argsort(np.argsort(allv)) + 1
n1, n2 = len(a), len(b)
U = ranks[:n1].sum() - n1*(n1+1)/2
print('合并排序秩 =', ranks.tolist(), '| U = %.0f（小值多在前 → U 小）' % U)
assert U == 2
print('✓ 非参数优势：只看相对大小（秩），抗离群/非正态；大样本 U 近正态可 z 化')

合并排序秩 = [1, 4, 3, 2, 5] | U = 2（小值多在前 → U 小）
✓ 非参数优势：只看相对大小（秩），抗离群/非正态；大样本 U 近正态可 z 化


In [41]:
# p 值/拒绝域可视化：z 分布 + 阴影面积 = α
xv = np.linspace(-4, 4, 300)
pdf = stats.norm.pdf(xv)
z0 = 1.96
plt.figure(figsize=(7, 3.0))
plt.plot(xv, pdf, lw=1.2, color='#1f4e79')
plt.fill_between(xv, pdf, where=(xv > z0), color='#b45309', alpha=0.45, label='右侧拒绝域 α/2')
plt.fill_between(xv, pdf, where=(xv < -z0), color='#b45309', alpha=0.45)
plt.axvline(z0, color='#b45309', ls='--', lw=1); plt.axvline(-z0, color='#b45309', ls='--', lw=1)
plt.text(z0+0.03, 0.28, 'z=1.96', color='#b45309', fontsize=9)
plt.text(-z0-0.42, 0.28, 'z=-1.96', color='#b45309', fontsize=9)
plt.xlabel('z'); plt.ylabel('密度'); plt.legend(fontsize=8)
plt.title('双侧检验：|z|>1.96 ⇔ p<0.05 → 拒绝 H0', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ 阴影面积=α；观察 z 落在阴影 → 拒绝；p 值 = 比观察值更极端的概率')

C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3742290546.py:14: UserWarning: Glyph 8660 (\N{LEFT RIGHT DOUBLE ARROW}) missing from font(s) Microsoft YaHei.
  plt.tight_layout(); plt.show()


✓ 阴影面积=α；观察 z 落在阴影 → 拒绝；p 值 = 比观察值更极端的概率


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\3742290546.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🍵 白话开篇：假设检验 = 法庭审判

想象法庭审案：**无罪推定**（H0：被告无罪），检方要拿出证据推翻。

- **H0（原假设）** = “无罪”：默认成立、没有证据不动摇；
- **证据** = 样本数据；**判决标准** = α（如 5%：证据强到“无罪情形下出现这种证据的概率 < 5%”才判有罪）；
- **p 值** = “假如被告真的无罪，出现比这更不利证据的概率”——p 小 = 证据强；
- **两类错误**：
  - 第一类（α）：无罪判有罪（冤枉好人）——**唯一受控**（设定 α）；
  - 第二类（β）：有罪判无罪（放过坏人）——靠功效 1−β 衡量。

**为什么“不显著”≠“没效应”**：证据不足只是“没到判罪标准”，不是“证明了无罪”。
（对应到业务：AB 不显著 ≠ 方案无效，可能样本不够 = 功效不足。）

**一句话**：假设检验 = 用概率衡量的“打脸 H0 的游戏”，只能拒绝、不能证明。

## 🍵 白话讲透中心极限定理（CLT）：摇骰子的奇迹

掷一次骰子：1~6 均匀分布，毫无“钟形”可言。但——**连掷 30 次取平均，重复 10000 次**：
这些“平均值”画成直方图，居然接近正态！这就是 CLT。

**CLT 说了什么（口头版）**：
不管单个数据是什么分布（偏的、双峰的、指数尾巴的），
只要**样本互相独立、样本量够大**，**样本均值的分布**就近似正态：
$$\bar x\ \dot{\sim}\ N\Big(\mu,\ \frac{\sigma^2}{n}\Big)$$

**逐符号**：$\bar x$：样本均值；$\mu$：总体均值；$\sigma^2/n$：均值抽样误差（n 越大越小）。

**为什么是统计学的地基**：
- t 检验/置信区间全靠它（均值的抽样分布已知）；
- 业务上：一个活动带来的日均新增 = 无数天波动的平均 → 天天正态 → 可做检验；
- AB 实验的“均值差”也是均值的差 → 照样正态 → 才能算 p 值。

**注意两个前提**：独立（同一个人别算两次）、n 够大（30+ 一般够，重尾要更多）。

**一句话**：正态不是数据的属性，是**均值**的属性——这是检验能算出来的底气。

## 🍵 白话讲透贝叶斯：用新证据更新“信念”

频率派说：参数是固定的，概率是长期频率。贝叶斯派说：**概率 = 我有多相信**，
先有一个信念（先验），来数据后更新（后验）。

$$P(\theta\mid x)=\frac{P(x\mid \theta)P(\theta)}{P(x)}$$

**逐符号解释（配比喻：猜硬币偏不偏）**：
- $\theta$：参数（硬币正面概率）；$x$：数据（扔 100 次的结果）；
- $P(\theta)$：**先验**——没扔之前我猜 θ 大概 0.5（均匀/弱先验）；
- $P(x|\theta)$：**似然**——如果 θ=0.7，出现这结果多像；
- $P(x)$：证据/归一化——所有 θ 下数据总概率，保证后验加起来=1；
- $P(\theta|x)$：**后验**——扔完 100 次之后我更新的信念。

**直觉（迭代更新）**：扔 20 次全是正面 → 后验往 1 靠；再扔 20 次正常 → 后验又拉回来。
**后验 ∝ 先验 × 似然**——先验是“历史记忆”，似然是“新证据”，二者相乘即更新。

**何时用贝叶斯**：样本少（先验救命）、要概率区间而非点估计（后验分布直接给）、
多轮更新（AB 序贯）。**代价**：后验难算 → MCMC/变分近似。

## 🔍 公式拆解 ①：单样本 t 检验

$$t=\frac{\bar x - \mu_0}{s/\sqrt{n}}$$

**逐符号解释**：
- $\bar x$：样本均值（观察到的平均）；$\mu_0$：H0 声称的总体均值（如“历史日均 100 单”）；
- $s$：样本标准差（数据的起伏）；$n$：样本量；
- $s/\sqrt{n}$：均值的**标准误（SE）**——“均值这个估计有多不稳”，n 越大越稳；
- 分子=“偏离 H0 多远”，分母=“这个偏离有多难置信”→ **t = 偏离的“信噪比”**。

**直觉**：t 像“身高差除以测量误差”——差得大且误差小 → 差异可信（t 大 → p 小）。

**为什么自由度 n−1**：估计 s 时用掉了 1 个自由度（均值固定后，最后 1 个数被锁定）→ 分布更“宽尾巴”
（小样本尤其明显；n→∞ 时 t → 标准正态）。

**数字验证**：$\bar x=12,\ \mu_0=10,\ s=4,\ n=16$ → $t=2/(4/4)=2.0$，
df=15 的临界值 ≈2.13 → |t|<2.13 不显著（这个例子里差点拒绝——感受下临界值尺度）。

## 🔍 公式拆解 ②：卡方检验

$$\chi^2=\sum_{i}\frac{(O_i-E_i)^2}{E_i},\qquad E_i=\frac{\text{行和}\times\text{列和}}{\text{总数}}$$

**逐符号解释**：
- $O_i$：第 i 个格子**观察到的频数**（如“男性点击 320 次”）；
- $E_i$：**期望频数**——H0（无关）下这个格子应该大约多少；
- $E_i$ 公式：两事件独立 → 联合概率 = 边际概率之积 → 频数 = 总数×行概率×列概率；
- 分子 $(O-E)^2$：偏差平方（正负都算）；除以 $E_i$：**按期望量级归一**（期望 10 差 5 严重，期望 1000 差 5 无所谓）。

**直觉（为什么要除以 E）**：比较“相对偏差”而非绝对偏差——
就像新闻点击率：1 万点击差 50 次和 100 点击差 50 次，后者离谱得多。

**自由度** = (行数−1)(列数−1)；**适用**：频数表、期望别太小（通常 E≥5）。

**数字小例**：期望 100 观察到 120 → 单项贡献 = 400/100 = 4；
所有格子加起来与临界值（df=1 时 χ²₀.₀₅≈3.84）比较——4>3.84 → 拒绝无关。

## 🔍 公式拆解 ③：F 检验（方差分析）

$$F=\frac{MS_B}{MS_W}=\frac{SS_B/(k-1)}{SS_W/(N-k)}$$

**逐符号解释**：
- $k$：组数（如 4 个版本）；$N$：总样本量；
- $SS_B$：**组间平方和**——各组均值离总体均值多远（“组与组差多少”）；
- $SS_W$：**组内平方和**——每个点离本组均值多远（“组内噪声多大”）；
- 除以各自自由度得均方 MS；$F$ = “组间差异 ÷ 组内噪声”。

**直觉**：F 大 = 组间差距明显大于组内起伏 → 说明至少有一组不同（拒绝 H0：全组均值相等）。

**与 t 的关系**：两组时 $F=t^2$（两检验等价）——这也是“先 F 后两两 t”的原因。

**注意**：F 显著 ≠ 知道哪组不同 → 后续要多重比较校正（Tukey/Bonferroni）。

**数字小例**：$SS_B=120,\ k=3,\ SS_W=150,\ N=15$ → $F=(120/2)/(150/12)=60/12.5=4.8$，
df=(2,12) 临界值 ≈3.89 → 4.8>3.89 拒绝。

## 🔍 公式拆解 ④：功效与样本量

$$1-\beta=\Phi\Big(\frac{|\Delta|}{\sigma\sqrt{2/n}}-z_{1-\alpha/2}\Big)$$

**逐符号解释**：
- $\Delta$：真实效应量（两均值差）；$\sigma$：标准差；$n$：每组样本量；
- $z_{1-\alpha/2}$：显著性临界值（α=0.05 双侧 → 1.96）；
- 第一项：真实差异在“标准误单位”下有多远——**效应越大、n 越大 → 越容易发现**；
- $\Phi$：标准正态 CDF——把“距离”转成“发现概率”。

**直觉**：功效 = “如果真有 Δ 的差异，实验能有多大概率测出来”。
像雷达：效应大（目标大）→ 容易发现；样本少（天线小）→ 容易漏。

**反解样本量（常用式）**：
$$n=\frac{2\sigma^2(z_{1-\alpha/2}+z_{1-\beta})^2}{\Delta^2}$$

- 想功效 80% → $z_{0.8}=0.84$；想 α=0.05 → 1.96；
- **平方效应**：Δ 减半 → n 要 ×4（这解释了为什么小效果实验那么贵）。

**一句话**：先定 α、β、Δ、σ 再算 n——上线前必做（本书 r9 有完整算例）。

In [42]:
# CLT 可视化：掷骰子均值的分布，随样本量从"三角"变"钟形"
rng = np.random.RandomState(7)
M = 20000
fig, ax = plt.subplots(1, 3, figsize=(9.5, 2.8))
for k, n in enumerate([2, 10, 50]):
    means = rng.randint(1, 7, size=(M, n)).mean(axis=1)
    ax[k].hist(means, bins=60, density=True, color='#1f4e79', alpha=0.75)
    s = np.sqrt((35/12)/n)                       # 骰子总体方差 = (6²-1)/12 = 35/12
    xs = np.linspace(2, 6, 200)
    ax[k].plot(xs, np.exp(-(xs-3.5)**2/(2*s**2))/np.sqrt(2*np.pi*s**2), 'r--', lw=1.5)
    ax[k].set_title('n=%d 的均值分布' % n, fontsize=9)
    ax[k].set_xlim(2, 6)
plt.tight_layout(); plt.show()
print('✓ n=2 还是三角；n=50 已紧贴正态曲线——均值分布正态化是检验可算的基础')
assert abs(means.mean() - 3.5) < 0.02

✓ n=2 还是三角；n=50 已紧贴正态曲线——均值分布正态化是检验可算的基础


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\522893066.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [43]:
# 两类错误可视化：H0 分布(蓝) vs H1 分布(红)，α/β/功效 = 阴影面积
mu0, mu1, sd, n = 100.0, 103.0, 15.0, 60
se = sd/np.sqrt(n)
x = np.linspace(92, 112, 400)
f0 = np.exp(-(x-mu0)**2/(2*se**2))/np.sqrt(2*np.pi*se**2)
f1 = np.exp(-(x-mu1)**2/(2*se**2))/np.sqrt(2*np.pi*se**2)
crit = mu0 + 1.645*se                              # 单侧 5% 临界
plt.figure(figsize=(7.5, 3.2))
plt.plot(x, f0, color='#1f4e79', lw=1.6, label='H0 真实 (μ=100)')
plt.plot(x, f1, color='#b45309', lw=1.6, label='H1 真实 (μ=103)')
plt.fill_between(x, 0, f0, where=x > crit, color='#1f4e79', alpha=0.25, label='α（冤枉好人）')
plt.fill_between(x, 0, f1, where=x < crit, color='#b45309', alpha=0.25, label='β（放过坏人）')
plt.axvline(crit, color='k', ls='--', lw=1)
plt.text(crit+0.15, 0.05, '临界值', fontsize=8, rotation=90)
plt.xlabel('样本均值'); plt.ylabel('密度')
plt.legend(fontsize=8); plt.title('α / β / 功效 = 1−β 都是面积', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ α 只由临界值决定（可控）；β 取决于真实效应与 n（不可控但可算）——提升 n 让两峰分开')

✓ α 只由临界值决定（可控）；β 取决于真实效应与 n（不可控但可算）——提升 n 让两峰分开


C:\Users\24260\AppData\Local\Temp\ipykernel_41256\2853370718.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⚠️ 本章高频踩坑（面试必查）

1. **p 值滥用**：p<0.05 ≠ “效应显著到值得上线”——还要看效应量和业务意义；
2. **多重比较**：跑 20 个指标，期望就有 1 个“假显著”——要校正（FDR/Bonferroni）；
3. **把不显著当“无效应”**：功效不足时不显著 = “没测出来”，不是“没有”；
4. **正态假设**：t/ANOVA 假设残差近正态+方差齐；重尾/异方差要用非参（Mann-Whitney 等）；
5. **χ² 条件**：期望频数过小（E<5）时检验失真——合并格子或 Fisher 精确检验；
6. **自由度算错**：χ²=(r−1)(c−1)，t= n−1，F 两组两套；
7. **方向性**：单侧 vs 双侧要先定（事后换方向 = 作弊）；
8. **数据窥探（p-hacking）**：先看数据再定假设 → 假阳性爆炸——假设先行。

## 🧭 三大学派对比：频率 / 贝叶斯 / 置换

| 学派 | 核心对象 | 输出 | 优点 | 缺点 |
|---|---|---|---|---|
| 频率派 | 长期频率下的检验统计量 | p 值、置信区间 | 客观、标准化 | 不能谈“参数的概率” |
| 贝叶斯 | 参数的信念分布 | 后验、可信区间 | 直观概率、能加先验 | 先验选择主观、计算贵 |
| 置换检验 | 数据本身的重排 | 精确 p 值 | 无分布假设 | 组合爆炸、样本小才实用 |

**选型逻辑**：
- 业务常规 → 频率派（简单、人人认）；
- 小样本/有先验/要概率解释 → 贝叶斯；
- 怀疑分布假设、样本中等 → 置换/自助（bootstrap）。

**一句话**：三派都回答“证据强不强”，只是对“概率”的定义不同——面试答出区别就赢。

## 🏁 本章学完清单

- [ ] 能用法庭比喻讲清 H0/α/β/功效与两类错误；
- [ ] 能口头证明 CLT 为什么是检验的地基；
- [ ] 能写 t/χ²/F 公式并逐符号解释；
- [ ] 能说出功效公式里每个变量怎么影响 n；
- [ ] 能徒手算：t 值、χ² 单项贡献、F 值（本书示例）；
- [ ] 能区分频率/贝叶斯/置换三派；
- [ ] 能说出 5 个 p 值使用红线；
- [ ] 会做：样本量反解、功效分析（r9 示例）。

> 下一站：AB 实验——把统计用到产品决策上（下一本 notebook）。